# Configurable HTG case study

A standalone case driver for **HTG_Model.ipynb**, based on the last full-text case in this discussion. It does not import or run HTG_Cases. Default: unrestricted gas carbon, strict continuation from the last successful lower temperature, 240 bar, an imposed 400 °C aqueous cut-off, aqueous N₂ and the expanded neutral-liquid candidates.

**Run order**
1. Edit the input cell below: file paths, operating conditions, feed, products, constraints and solver controls.
2. Run the model loader, case preparation and preflight.
3. Run the sweep. Review `diagnostics`, especially failed points and seed temperatures.
4. Run the result tables and plots. The final code cell is an optional Figure 7–8 comparison.

After changing physical inputs, rerun preparation, preflight and the sweep. Changing only plot controls requires rerunning the plotting cells. Reload the model after changing database paths or species overrides.

**Files required:** the updated HTG_Model(2).ipynb with gas-carbon inequality and SUPCRT mineral support; NASA thermochemistry; PC-SAFT CSV; ePC-SAFT CSV; SUPCRT XML. Use the same Python/Julia environment as the working model (including pyclapeyron, juliacall and pychnosz). No other case notebook or Excel workbook is required.

In [ ]:
from pathlib import Path
from pprint import pprint
import copy
import hashlib
import json
import re
import time
import types
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

## Inputs — edit this cell

Every editable input has a comment with its meaning and units. Species lists are independent: adding a molecule to one phase does not automatically add it to another. Use exact names from the loaded species inventory. All configured phase species remain candidates; zero feed inventory will force species containing absent elements to zero.

| Input group | Meaning |
|---|---|
| Temperature / pressure | Temperatures in °C; absolute pressure in bar. The grid is sorted increasingly for continuation. Optional pressure overrides apply at exactly matching temperature keys. |
| Aqueous cut-off | A modelling rule, applied at **T ≥ cut-off**, independent of pressure. `None` leaves aqueous available; `AQUEOUS_ENABLED=False` removes it everywhere. It does not calculate a mixture critical curve. |
| UCA | Elemental mass percentages on a dry or dry-ash-free basis. Oxygen must be entered. Ash is included in the dry-feed denominator and is outside the reactive model unless its constituent elements/minerals are explicitly represented. |
| Water | Feed moisture is wt% as received. Process water is g/g original dry feed. Moisture water and process water are added once, separately from dry UCA. |
| Salt additions | mol salt/g original dry feed. These add elemental inventories. Their starting chemical identity does not constrain the products. |
| Carbon mode | `unrestricted`, `minimum` or `exact`. The target is a fraction of **all feed carbon including salt carbon**, not a gas concentration. Additional phase-carbon constraints, if supplied, still apply. |
| Consecutive solver | `True`: only the last successful compatible lower-T result seeds the next point. `False`: each point starts from a fresh balanced LP. No competing independent start is used in consecutive mode. |
| Solvent floor / bounds | Amounts in mol/mol original dry-feed C. The aqueous solvent floor disappears with the aqueous phase. |
| Optimizer | SLSQP controls; `ftol` applies to the scaled objective. A successful exit alone does not prove a global minimum. |

**Default feed:** the reduced C/H/O/N/S dry-manure inventory used previously. The 8.44688764% remainder is assigned to unmodelled ash for bookkeeping. This is a modelling assumption, not a measured ash analysis. The small NaHCO₃/KCl/MgCO₃ additions are retained from the last test (0.01 mol/kg dry feed each).

**Default products:** N₂ is allowed in gas and aqueous. Benzene, toluene, methanol, ethanol and acetone compete in all three fluid phases. The separate organic liquid currently excludes water, so water/oil partitioning remains restricted. The selected organics are a limited surrogate, not a complete biocrude composition. The model uses pure solids, not a solid solution. The XML hydrate data and their equation limits are checked; no external hydrate records are added.

In [ ]:
# ===================== FILES AND DATABASES =====================
# Paths are relative to the directory where Jupyter is running.
# Prefer explicit paths if you keep several model/database versions.
MODEL_NOTEBOOK = Path('HTG_Model(2).ipynb')
DATABASE_DIRECTORY = Path('.')
DATABASE_FILENAMES = {
    'nasa': ['Thermdat_WasteMech.txt'],
    'pcsaft': ['PROJECT_PCSAFT.csv'],
    'epcsaft': ['PROJECT_EPCSAFT.csv'],
    'supcrt': ['supcrt92.xml'],
}
# Exact isomer mapping. Do not match distinct molecules by formula alone.
SPECIES_OVERRIDES = {'acetone': {'nasa': 'CH3COCH3'}}

# ====================== OPERATING CONDITIONS ====================
CASE_NAME = 'full_test_configurable'
TEMPERATURES_C = [100, 150, 200, 250, 300, 340, 360, 380, 389.6,
                  390, 400, 420, 460, 500, 540, 580]
PRESSURE_BAR = 240.0  # Absolute pressure, not gauge pressure.
# Optional absolute pressure overrides at individual temperatures.
PRESSURE_BY_T_C = {}  # Example: {400.0: 250.0, 420.0: 250.0}

# Default 400 C follows the last discussed test-only override.
# Set any temperature here; 390 C reproduces the original imposed cut-off.
AQUEOUS_CUTOFF_C = 385.0  # Aqueous removed at T >= cutoff; None = no cutoff.
AQUEOUS_ENABLED = True   # False = remove aqueous at EVERY temperature.

# Optional extra points near phase removal; None disables refinement.
REFINEMENT = {'start_C': 300.0, 'stop_C': 400.0, 'step_C': 5.0}

# ======================== FEEDSTOCK UCA =========================
# UCA = ultimate chemical analysis. Values are ELEMENTAL mass percentages.
# 'dry': percentages per 100 g dry feed, including ash in the denominator.
# 'dry_ash_free': elemental percentages must sum to 100; converted using ash.
# Oxygen is entered explicitly, not calculated automatically by difference.
UCA_BASIS = 'dry'
UCA_WT_PERCENT = {
    'C': 30.3950366, 
    'H': 4.40810496, 
    'O': 53.57873112,
    'N': 2.66469168, 
    'S': 0.506548,
    # Add other modelled elements here, e.g. 'Cl': 0.1.
}
ASH_DRY_WT_PERCENT = 8.44688764  # Not a model species; see documentation.
# Report any additional unmodelled dry fraction explicitly; default none.
UNMODELLED_DRY_WT_PERCENT = 0.0

# Moisture is separate from dry UCA; entered as % of as-received wet feed.
FEED_MOISTURE_AR_WT_PERCENT = 0.0
# Additional process water, g water/g ORIGINAL dry feed (not including salts).
PROCESS_WATER_G_PER_G_DRY_FEED = 3.999935304

# Optional alternative to UCA: dry species in relative molar amounts.
# If not None, this replaces UCA; ash/unmodelled fractions must then be zero.
FEED_SPECIES_MOLES = None  # Example: {'cellulose': 1.0}

# Additions are mol of salt/g ORIGINAL dry feed; 1e-5 = 0.01 mol/kg dry feed.
# They add atoms, not prescribed products, and do not renormalize the dry feed.
ADDED_SALTS = {
    'NaHCO3': {'mol_per_g_dry_feed': 1e-5, 'elements': {'Na':1, 'H':1, 'C':1, 'O':3}},
    'KCl': {'mol_per_g_dry_feed': 1e-5, 'elements': {'K':1, 'Cl':1}},
    'MgCO3': {'mol_per_g_dry_feed': 1e-5, 'elements': {'Mg':1, 'C':1, 'O':3}},
    'CsCl': {'mol_per_g_dry_feed': 0.0, 'elements': {'Cs':1, 'Cl':1}},
    'LiCl': {'mol_per_g_dry_feed': 0.0, 'elements': {'Li':1, 'Cl':1}},
    'MgCl2': {'mol_per_g_dry_feed': 0.0, 'elements': {'Mg':1, 'Cl':2}},
}

# ==================== ALLOWED PRODUCTS BY PHASE =================
# Exact model registry names, case sensitive. Edit each list independently.
# Adding a species permits it; it does not force a positive equilibrium yield.
# Ions belong ONLY in aqueous. Aqueous requires 'water'.
PHASE_SPECIES = {
    'gas': [
        'water', 'hydrogen', 'carbon dioxide', 'carbon monoxide', 'methane',
        'ammonia', 'hydrogen sulfide', 'nitrogen',
        'benzene', 'toluene', 'methanol', 'ethanol', 'acetone',
    ],
    'aqueous': [
        'water', 'hydrogen', 'carbon dioxide', 'carbon monoxide', 'methane',
        'ammonia', 'hydrogen sulfide', 'nitrogen',
        'benzene', 'toluene', 'methanol', 'ethanol', 'acetone',
        'Na+', 'K+', 'Mg2+', 'Cl-', 'HCO3-', 'CO3^2-',
    ],
    'organic liquid': ['benzene', 'toluene', 'methanol', 'ethanol', 'acetone'],
    'solid': [
        'carbon', 'Artinite', 'Hydromagnesite', 'Brucite', 'Magnesite',
        'Periclase', 'Halite', 'Sylvite', 'Sodium-Oxide', 'Potassium-Oxide',
    ],
}

# ===================== CARBON ALLOCATION ========================
# 'unrestricted': no prescribed gas-carbon fraction.
# 'minimum': gas carbon >= GAS_CARBON_FRACTION of TOTAL feed carbon.
# 'exact': gas carbon == GAS_CARBON_FRACTION of TOTAL feed carbon.
# These are carbon-atom fractions, not gas mole fractions or gas mass yields.
CARBON_MODE = 'unrestricted'
GAS_CARBON_FRACTION = 0.90  # Used only for 'minimum' or 'exact'.
SOLID_CARBON_FRACTION = None  # Optional exact solid/char-carbon fraction.
SOLID_CARBON_TARGET = 'CHAR'  # 'CHAR' = carbon only; 'all solids' includes minerals.
PHASE_CARBON_FRACTIONS = {}   # Optional exact fractions, e.g. {'organic liquid':0.05}.

# ====================== SOLVER / STARTING GUESS =================
RUN_CONSECUTIVE = True
# True: use ONLY the last successful lower-T solution when selections match.
# False: use a new feasible LP initialization at EVERY temperature.
# At phase removal, a fresh LP is required. Failed points retain the last
# successful seed; there is no independent fallback when continuation fails.

# First point / new phase selection / nonconsecutive runs: LP starting side.
# 'auto' = liquid below cutoff, vapor when aqueous is disabled.
INITIALIZATION_DEFAULT = 'auto'  # 'auto', 'liquid', or 'vapor'.
INITIALIZATION_BY_T_C = {}  # Example: {380.0:'liquid', 390.0:'vapor'}.
INITIAL_GAS_WATER_FRACTION = 1e-6  # Liquid-start gas-water cap, not a final constraint.
AQUEOUS_MODEL = 'electrolyte'     # 'electrolyte', or 'molecular' without ions.
AQUEOUS_WATER_FLOOR = 1e-6       # mol water/mol original dry-feed C, while aqueous exists.
MINIMUM_AMOUNTS = {}             # {(phase, species): mol/mol original dry-feed C}.
MAX_ITERATIONS = 5000
OPTIMIZER_FTOL = 1e-9            # SLSQP tolerance on the scaled objective.
OPTIMIZER_EPS = 1e-7             # Finite-difference step in mol/mol dry-feed C.
OPTIMIZER_ENERGY_SCALE = None    # None = model default max(10000, abs(initial G)).
BALANCE_TOLERANCE = 1e-7         # Acceptance tolerance on normalized atom/charge balances.

# ============================= PLOTS ============================
PLOT_TOP_SPECIES = 12           # None = all species above the threshold.
PLOT_MIN_YIELD = 1e-8           # mol/kg original dry feed.
SPECIES_Y_SCALE = 'symlog'      # 'linear' or 'symlog'; zeros remain visible.
SAVE_FIGURES = False
FIGURE_DIRECTORY = Path('htg_figures')  # Used only when SAVE_FIGURES=True.

## Load the model and inspect database coverage

The model notebook is executed once as a module; this notebook supplies only case handling, initial guesses, diagnostics and plots. It retains the model's chemical potentials, EOS formulations and NASA/HKF reference alignment. Exact file paths are printed so that an older model copy is not loaded silently. If the inequality capability check fails, select the updated model notebook rather than bypassing the check.

The acetone mapping is explicit because propanal has the same elemental formula. To find further species, inspect `available_species` or `species_catalogue`; a NASA record alone does not provide a SAFT model.

In [ ]:
def locate_database(names):
    for folder in (DATABASE_DIRECTORY, DATABASE_DIRECTORY / 'upload'):
        for name in names:
            path = folder / name
            if path.is_file():
                return path.resolve()
    raise FileNotFoundError(f'No file found for {names}. Edit DATABASE_DIRECTORY/DATABASE_FILENAMES.')


def load_model_notebook(path):
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(f'Model not found: {path}. Set MODEL_NOTEBOOK to your actual filename.')
    document = json.loads(path.read_text(encoding='utf-8'))
    module = types.ModuleType('htg_model')
    for index, cell in enumerate(document['cells']):
        if cell['cell_type'] == 'code':
            exec(compile(''.join(cell['source']), f'{path}:cell{index+1}', 'exec'), module.__dict__)
    return module


DATABASE_PATHS = {key: locate_database(names) for key, names in DATABASE_FILENAMES.items()}
print('Model:', MODEL_NOTEBOOK.resolve())
for key, path in DATABASE_PATHS.items():
    print(f'{key}: {path}')
model = load_model_notebook(MODEL_NOTEBOOK)
if not getattr(model, 'SUPPORTS_GAS_CARBON_MIN_FRACTION', False):
    raise RuntimeError('This model predates the gas-carbon inequality. Use the updated HTG_Model(2).ipynb.')
for required in ('register_supcrt_minerals', 'prepare_phase_allocation'):
    if not hasattr(model, required):
        raise RuntimeError(f'Model lacks {required}; use the updated model file.')
available_species = model.initialize_model(DATABASE_PATHS, species_overrides=SPECIES_OVERRIDES)
species_catalogue = model.available_species(include_unavailable=True)
print('Ready species by registry category:')
display(available_species.groupby('Phase').size().rename('Count').to_frame())
# To inspect all candidates, uncomment:
# display(available_species)
# display(species_catalogue[~species_catalogue.Ready])
# display(model.mineral_inventory(elements=['Na','K','Mg','Li','Cs']))

provenance = pd.DataFrame([
    {'Kind':'model', 'Path':str(MODEL_NOTEBOOK.resolve()),
     'SHA256':hashlib.sha256(MODEL_NOTEBOOK.read_bytes()).hexdigest()},
    *[{'Kind':key,'Path':str(path),'SHA256':hashlib.sha256(path.read_bytes()).hexdigest()}
      for key,path in DATABASE_PATHS.items()]
])
display(provenance)

## Prepare the case and continuation behaviour

The UCA is converted to dry elemental wt% without silently normalizing an incomplete analysis. The feed must contain carbon because the underlying solver normalizes amounts to dry-feed carbon. Dry UCA + ash + unmodelled fraction must sum to 100% (within 0.05 percentage points). For dry-ash-free input, elemental percentages must sum to 100% before multiplication by the organic dry fraction. Species-defined feed overrides UCA and requires zero ash/unmodelled fractions.

The local phase-allocation wrapper preserves compatible initial compositions above the cut-off. A populated disappearing phase is never simply deleted from a guess: a new balanced LP start is requested. This wrapper is installed automatically after model loading and makes no change to Gibbs-energy formulas.

In [ ]:
def finite_number(value, name, low=None, high=None):
    value = float(value)
    if not np.isfinite(value) or (low is not None and value < low) or (high is not None and value > high):
        raise ValueError(f'Invalid {name}: {value}')
    return value


def prepare_feed():
    ash = finite_number(ASH_DRY_WT_PERCENT, 'dry ash wt%', 0, 100)
    unmodelled = finite_number(UNMODELLED_DRY_WT_PERCENT, 'unmodelled dry wt%', 0, 100)
    if FEED_SPECIES_MOLES is not None:
        if ash != 0 or unmodelled != 0:
            raise ValueError('For species-defined feed, set ash and unmodelled fractions to zero.')
        dry_uca = model.feed_from_species(FEED_SPECIES_MOLES)
    else:
        analysis = {e: finite_number(v, f'UCA {e}', 0, 100) for e,v in UCA_WT_PERCENT.items()}
        if UCA_BASIS == 'dry':
            dry_uca = analysis
        elif UCA_BASIS == 'dry_ash_free':
            if unmodelled != 0:
                raise ValueError('Dry-ash-free conversion requires UNMODELLED_DRY_WT_PERCENT=0.')
            if not np.isclose(sum(analysis.values()), 100, atol=.05, rtol=0):
                raise ValueError('Dry-ash-free elemental UCA must sum to 100%.')
            dry_uca = {e: v*(1-ash/100) for e,v in analysis.items()}
        else:
            raise ValueError('UCA_BASIS must be dry or dry_ash_free.')
        if not np.isclose(sum(dry_uca.values())+ash+unmodelled, 100, atol=.05, rtol=0):
            raise ValueError('Dry elemental UCA + dry ash + unmodelled fraction must sum to 100%. No automatic renormalization is applied.')
    if sum(dry_uca.values()) > 100+1e-8:
        raise ValueError('Dry elemental UCA exceeds 100 wt%.')
    if dry_uca.get('C',0) <= 0:
        raise ValueError('The carbon-normalized model requires positive dry-feed carbon.')
    if set(dry_uca)-set(model.AW):
        raise ValueError(f'Missing atomic weights: {set(dry_uca)-set(model.AW)}')
    moisture = finite_number(FEED_MOISTURE_AR_WT_PERCENT, 'as-received moisture', 0)
    if moisture >= 100:
        raise ValueError('As-received moisture must be <100 wt%.')
    moisture_water = moisture/(100-moisture)
    process_water = finite_number(PROCESS_WATER_G_PER_G_DRY_FEED, 'process water ratio', 0)
    return dry_uca, moisture_water, process_water+moisture_water


def make_temperature_grid():
    grid = [finite_number(t, 'temperature C', -273.149999) for t in TEMPERATURES_C]
    if not grid:
        raise ValueError('Enter at least one temperature.')
    low, high = min(grid), max(grid)
    if REFINEMENT is not None:
        step = finite_number(REFINEMENT['step_C'], 'refinement step', 1e-12)
        a = max(low, float(REFINEMENT['start_C']))
        b = min(high, float(REFINEMENT['stop_C']))
        if b >= a:
            grid += list(np.arange(a,b+step*1e-8,step))
    if AQUEOUS_CUTOFF_C is not None:
        cutoff = finite_number(AQUEOUS_CUTOFF_C, 'aqueous cutoff', -273.149999)
        if low <= cutoff <= high:
            grid.append(cutoff)
    return sorted(set(round(float(t),8) for t in grid))


def build_case_settings():
    dry_uca, moisture_water, total_water = prepare_feed()
    grid = make_temperature_grid()
    pressure = finite_number(PRESSURE_BAR, 'absolute pressure', 1e-12)
    if CARBON_MODE not in ('unrestricted','minimum','exact'):
        raise ValueError('CARBON_MODE must be unrestricted, minimum or exact.')
    fraction = (finite_number(GAS_CARBON_FRACTION, 'gas carbon fraction', 0, 1)
                if CARBON_MODE != 'unrestricted' else None)
    if AQUEOUS_MODEL not in ('electrolyte','molecular'):
        raise ValueError('AQUEOUS_MODEL must be electrolyte or molecular.')
    if INITIALIZATION_DEFAULT not in ('auto','liquid','vapor'):
        raise ValueError('INITIALIZATION_DEFAULT must be auto, liquid or vapor.')
    if set(PHASE_SPECIES)-{'gas','aqueous','organic liquid','solid'}:
        raise ValueError('Unknown phase in PHASE_SPECIES.')
    finite_number(OPTIMIZER_FTOL, 'optimizer ftol', 1e-16)
    finite_number(OPTIMIZER_EPS, 'optimizer eps', 1e-16)
    finite_number(BALANCE_TOLERANCE, 'balance tolerance', 1e-16)
    finite_number(AQUEOUS_WATER_FLOOR, 'aqueous water floor', 0)
    finite_number(INITIAL_GAS_WATER_FRACTION, 'initial gas water fraction', 0)
    if not isinstance(MAX_ITERATIONS, (int,np.integer)) or MAX_ITERATIONS <= 0:
        raise ValueError('MAX_ITERATIONS must be a positive integer.')
    if SPECIES_Y_SCALE not in ('linear','symlog'):
        raise ValueError('SPECIES_Y_SCALE must be linear or symlog.')
    finite_number(PLOT_MIN_YIELD,'plot minimum yield',1e-300)
    if PLOT_TOP_SPECIES is not None and (not isinstance(PLOT_TOP_SPECIES,(int,np.integer)) or PLOT_TOP_SPECIES<=0):
        raise ValueError('PLOT_TOP_SPECIES must be a positive integer or None.')
    settings = dict(
        pressure_bar=pressure, temperature_C=grid[0],
        feed_element_wt_percent=dry_uca, feed_species_moles=None,
        water_g_per_g_dry_feed=total_water, added_salts=copy.deepcopy(ADDED_SALTS),
        phase_species=copy.deepcopy(PHASE_SPECIES),
        aqueous_cutoff_C=AQUEOUS_CUTOFF_C, aqueous_enabled=AQUEOUS_ENABLED,
        gas_carbon_fraction=fraction if CARBON_MODE=='exact' else None,
        gas_carbon_min_fraction=fraction if CARBON_MODE=='minimum' else None,
        solid_carbon_fraction=SOLID_CARBON_FRACTION,
        solid_carbon_target=SOLID_CARBON_TARGET,
        phase_carbon_fractions=copy.deepcopy(PHASE_CARBON_FRACTIONS),
        aqueous_model=AQUEOUS_MODEL, aqueous_water_floor=AQUEOUS_WATER_FLOOR,
        minimum_amounts=copy.deepcopy(MINIMUM_AMOUNTS),
        initial_phase_amounts=None, initial_gas_water_fraction=INITIAL_GAS_WATER_FRACTION,
        max_iterations=MAX_ITERATIONS, optimizer_ftol=OPTIMIZER_FTOL,
        optimizer_eps=OPTIMIZER_EPS,
    )
    if OPTIMIZER_ENERGY_SCALE is not None:
        settings['optimizer_energy_scale'] = finite_number(OPTIMIZER_ENERGY_SCALE,'energy scale',1e-12)
    if CARBON_MODE != 'unrestricted' and 'gas' in PHASE_CARBON_FRACTIONS:
        raise ValueError('Do not combine gas CARBON_MODE with an extra gas phase-carbon equality.')
    if CARBON_MODE == 'unrestricted' and 'gas' in PHASE_CARBON_FRACTIONS:
        raise ValueError('Remove the gas entry in PHASE_CARBON_FRACTIONS for unrestricted gas carbon.')
    if not settings['phase_species'].get('gas'):
        raise ValueError('Select gas species for this case and its gas-carbon diagnostics.')
    for name, salt in settings['added_salts'].items():
        finite_number(salt['mol_per_g_dry_feed'],f'salt addition {name}',0)
        for e, count in salt['elements'].items():
            if e not in model.AW:
                raise ValueError(f'Missing atomic weight for {e}')
            finite_number(count,f'{name} element count {e}',0)
    summary = {'Case':CASE_NAME,'Carbon mode':CARBON_MODE,'Consecutive':RUN_CONSECUTIVE,
               'Cutoff_C':AQUEOUS_CUTOFF_C,'Aqueous_enabled':AQUEOUS_ENABLED,
               'Moisture_water_g_per_g_dry':moisture_water,
               'Total_added_water_g_per_g_dry':total_water,'Dry_UCA':dry_uca}
    return settings, grid, summary


# Preserve supplied compatible starts even when aqueous is already disabled.
# The underlying model remains responsible for all balances and thermodynamics.
def install_continuation_allocator(module):
    if not hasattr(module,'_case_driver_original_allocator'):
        module._case_driver_original_allocator = module.prepare_phase_allocation
    original = module._case_driver_original_allocator
    def allocator(settings):
        request = copy.deepcopy(settings)
        initial = request.get('initial_phase_amounts')
        request['initial_phase_amounts'] = None
        case, report = original(request)
        report['initial_composition_reset'] = False
        if initial is not None:
            if any(not np.isfinite(v) or v < 0 for v in initial.values()):
                raise ValueError('Invalid initial amount.')
            selected = {(p,s) for p,names in case['phase_species'].items() for s in names}
            lost = [pair for pair,v in initial.items() if pair not in selected and v > 0]
            if lost:
                report['initial_composition_reset'] = True
                report['initial_reset_reason'] = f'Populated species removed: {lost}'
            else:
                case['initial_phase_amounts'] = {pair:v for pair,v in initial.items() if pair in selected}
        return case, report
    module.prepare_phase_allocation = allocator


install_continuation_allocator(model)
BASE_SETTINGS, temperature_grid, case_definition = build_case_settings()
registry = {**model.ALL_MOLECULAR_DATA, **model.SOLID_DATA, **model.ION_DATA}
print('Effective case definition:')
pprint(case_definition, sort_dicts=False)
print('Temperature grid:', temperature_grid)
display(pd.DataFrame([{'Phase':p,'Species':s,'Formula':registry.get(s,{}).get('formula','UNKNOWN')}
                      for p,names in PHASE_SPECIES.items() for s in names]))

## Preflight — before any EOS equilibrium calculation

At every temperature this checks registered phase/species names, ionic data support, NASA/SAFT parameter coverage, solid equation temperature limits and the **full nonnegative elemental/charge/carbon balance system**. This includes the minimum-carbon inequality when selected. A missing carrier (for example a nonzero Li/Cs feed without a supported product) is an input error; atoms are not dropped. Database coverage and a feasible atom balance do not establish mixture-EOS validity or equilibrium convergence.

In [ ]:
def point_settings(temperature):
    case = copy.deepcopy(BASE_SETTINGS)
    case['temperature_C'] = float(temperature)
    case['pressure_bar'] = finite_number(
        PRESSURE_BY_T_C.get(float(temperature), PRESSURE_BAR), 'point pressure', 1e-12)
    cutoff = case.get('aqueous_cutoff_C')
    disabled = case.get('aqueous_enabled') is False or (cutoff is not None and temperature >= cutoff)
    mode = INITIALIZATION_BY_T_C.get(float(temperature), INITIALIZATION_DEFAULT)
    if mode == 'auto':
        mode = 'vapor' if disabled else 'liquid'
    if mode not in ('liquid','vapor'):
        raise ValueError(f'{temperature} C: invalid initialization {mode}')
    case['liquid_initialization'] = not disabled and mode == 'liquid'
    case['gas_water_initialization'] = disabled or mode == 'vapor'
    return case


def feed_atoms_per_g(case):
    atoms = {e: wt/100/model.AW[e] for e,wt in case['feed_element_wt_percent'].items()}
    water = case['water_g_per_g_dry_feed']/(2*model.AW['H']+model.AW['O'])
    atoms['H'] = atoms.get('H',0)+2*water
    atoms['O'] = atoms.get('O',0)+water
    for salt in case['added_salts'].values():
        for e,count in salt['elements'].items():
            atoms[e] = atoms.get(e,0)+salt['mol_per_g_dry_feed']*count
    return atoms


def check_point(case):
    effective, allocation = model.prepare_phase_allocation(case)
    pairs = [(p,s) for p,names in effective['phase_species'].items() for s in names]
    if not pairs:
        raise ValueError('No product species selected.')
    ready_ions = set(available_species.loc[available_species.Phase.eq('aqueous ion'),'Species'])
    for phase,names in effective['phase_species'].items():
        if len(names) != len(set(names)):
            raise ValueError(f'Duplicate species in {phase}')
        for species in names:
            if phase == 'solid':
                if species not in model.SOLID_DATA:
                    raise ValueError(f'Unregistered solid: {species}')
                entry = model.SOLID_DATA[species]
                if entry.get('adapter') == 'supcrt_mineral':
                    model.supcrt_mineral_properties(species,case['temperature_C']+273.15,case['pressure_bar']*1e5)
                else:
                    model.nasa7_properties(model.nasa_db,entry['nasa'],case['temperature_C']+273.15)
            elif species in model.ION_DATA:
                if phase != 'aqueous' or AQUEOUS_MODEL != 'electrolyte' or species not in ready_ions:
                    raise ValueError(f'{species}: requires complete ionic EOS/HKF data and aqueous electrolyte phase.')
            else:
                if species not in model.ALL_MOLECULAR_DATA:
                    raise ValueError(f'Unregistered molecule: {species}')
                entry = model.ALL_MOLECULAR_DATA[species]
                model.resolve_molecular_parameters(case['temperature_C']+273.15,[species])
                model.nasa7_properties(model.nasa_db,entry['nasa'],case['temperature_C']+273.15)
    if effective['phase_species'].get('aqueous') and 'water' not in effective['phase_species']['aqueous']:
        raise ValueError('Aqueous requires water.')
    if effective['phase_species'].get('aqueous') and AQUEOUS_MODEL == 'electrolyte' and AQUEOUS_WATER_FLOOR <= 1e-9:
        raise ValueError('Electrolyte solvent floor must exceed 1e-9 mol/mol dry-feed C.')
    atoms = feed_atoms_per_g(case)
    elements = sorted(set(atoms) | {e for p,s in pairs for e in registry[s]['elements']})
    matrix = np.asarray([[registry[s]['elements'].get(e,0) for p,s in pairs] for e in elements],dtype=float)
    targets = np.asarray([atoms.get(e,0) for e in elements])
    missing = [e for e,b,row in zip(elements,targets,matrix) if b > 0 and not np.any(row > 0)]
    if missing:
        raise ValueError('No product carrier for supplied elements: '+', '.join(missing))
    charge = np.asarray([model.ION_DATA[s]['charge'] if p=='aqueous' and s in model.ION_DATA else 0 for p,s in pairs])
    carbon = matrix[elements.index('C')]
    gas_carbon = carbon*np.asarray([p=='gas' for p,s in pairs])
    eq, rhs = list(matrix)+[charge], list(targets)+[0.]
    if case['gas_carbon_fraction'] is not None:
        eq.append(gas_carbon);rhs.append(case['gas_carbon_fraction']*atoms['C'])
    for phase,fraction in case['phase_carbon_fractions'].items():
        if phase not in ('gas','aqueous','organic liquid','solid'):
            raise ValueError(f'Unknown carbon-allocation phase: {phase}')
        fraction = finite_number(fraction,f'{phase} carbon fraction',0,1)
        eq.append(carbon*np.asarray([p==phase for p,s in pairs]));rhs.append(fraction*atoms['C'])
    if case['solid_carbon_fraction'] is not None:
        fraction = finite_number(case['solid_carbon_fraction'],'solid carbon fraction',0,1)
        target = case['solid_carbon_target']
        if target not in ('CHAR','all solids'):
            raise ValueError('SOLID_CARBON_TARGET must be CHAR or all solids.')
        if target=='CHAR' and ('solid','carbon') not in pairs:
            raise ValueError('Select solid carbon to constrain CHAR.')
        eq.append(carbon*np.asarray([p=='solid' and (target=='all solids' or s=='carbon') for p,s in pairs]))
        rhs.append(fraction*atoms['C'])
    dry_C = case['feed_element_wt_percent']['C']/100/model.AW['C']
    bounds = [(0.,None)]*len(pairs)
    for pair,value in effective.get('minimum_amounts',{}).items():
        if pair not in pairs:
            raise ValueError(f'Lower-bound species not selected: {pair}')
        bounds[pairs.index(pair)] = (finite_number(value,'minimum amount',0)*dry_C,None)
    if ('aqueous','water') in pairs:
        j = pairs.index(('aqueous','water'))
        bounds[j] = (max(bounds[j][0],case['aqueous_water_floor']*dry_C),None)
    inequality = {}
    if case['gas_carbon_min_fraction'] is not None:
        inequality = {'A_ub':-gas_carbon[None,:], 'b_ub':[-case['gas_carbon_min_fraction']*atoms['C']]}
    feasible = model.linprog(np.zeros(len(pairs)),A_eq=np.asarray(eq),b_eq=np.asarray(rhs),
                            bounds=bounds,method='highs',**inequality)
    if not feasible.success:
        raise ValueError('Full elemental/charge/carbon constraints are infeasible: '+feasible.message)
    return effective, allocation


preflight_rows = []
for temperature in temperature_grid:
    case = point_settings(temperature)
    try:
        effective, allocation = check_point(case)
        preflight_rows.append({'T_C':temperature,'P_bar':case['pressure_bar'],
                              'Aqueous_disabled':allocation['aqueous_disabled'],
                              'Status':'ready','Error':''})
    except Exception as exc:
        preflight_rows.append({'T_C':temperature,'P_bar':case['pressure_bar'],
                              'Aqueous_disabled':np.nan,'Status':'invalid','Error':str(exc)})
preflight = pd.DataFrame(preflight_rows)
with pd.option_context('display.max_rows',None,'display.max_colwidth',None):
    display(preflight)
if not preflight.Status.eq('ready').all():
    raise ValueError('Fix invalid inputs/species coverage before running the solver. No feed atoms have been discarded.')
print('Preflight passed: thermochemical/parameter coverage and full balance feasibility. Mixture EOS validity is evaluated during solving.')

_validated_point_settings = {t:repr(point_settings(t)) for t in temperature_grid}

## Solve the temperature sweep

Consecutive mode always uses the **last successful lower-temperature result**, provided the selected phase/species pairs match. There is no lower-Gibbs competition with an independent start. A failed point remains a gap, and the next point keeps the last successful seed. At aqueous removal the species selection changes, so the driver uses a new balanced LP start; continuation resumes after the first successful point with the new phase selection.

The reported `Start`, `Seed_T_C`, balance/charge residuals, iterations and EOS messages let you verify what actually happened. This heating path can follow a local or metastable branch; smooth profiles do not prove global equilibrium. To run independent points, set RUN_CONSECUTIVE=False and rerun this cell after any required preparation/preflight. Outputs are not precomputed or presented as validated thermodynamic predictions.

In [ ]:
# Require validated inputs; detect edits made since the preparation/preflight cells.
if 'preflight' not in globals() or not preflight.Status.eq('ready').all():
    raise RuntimeError('Run the preparation and preflight cells successfully first.')

candidate_base, candidate_grid, candidate_definition = build_case_settings()
if repr(candidate_base)!=repr(BASE_SETTINGS) or candidate_grid!=temperature_grid:
    raise RuntimeError('Inputs changed. Rerun case preparation and preflight before solving.')
if any(repr(point_settings(t))!=_validated_point_settings[t] for t in temperature_grid):
    raise RuntimeError('Point pressure or initialization changed. Rerun preflight.')
run_grid = list(temperature_grid)
run_config = copy.deepcopy(BASE_SETTINGS)
run_name = str(CASE_NAME)
run_controls = {'consecutive':bool(RUN_CONSECUTIVE),'carbon_mode':CARBON_MODE,
                'gas_fraction':GAS_CARBON_FRACTION}
case_results, diagnostic_rows = [], []
previous = None
start_time = time.perf_counter()
for temperature in temperature_grid:
    case = point_settings(temperature)
    effective, allocation = model.prepare_phase_allocation(case)
    signature = {(p,s) for p,names in effective['phase_species'].items() for s in names}
    compatible = previous is not None and set(previous['amounts'])==signature
    use_previous = bool(RUN_CONSECUTIVE and compatible)
    if use_previous:
        case['initial_phase_amounts'] = copy.deepcopy(previous['amounts'])
        start_kind = 'previous successful lower temperature'
    else:
        case['initial_phase_amounts'] = None
        start_kind = ('fresh LP: phase/species selection changed'
                      if RUN_CONSECUTIVE and previous is not None and not compatible
                      else 'fresh LP')
    row = {'T_C':temperature,'P_bar':case['pressure_bar'], 'Status':'failed',
           'Start':start_kind,'Seed_T_C':previous['temperature_C'] if use_previous else np.nan,
           'Aqueous_disabled':allocation['aqueous_disabled'], 'Error':'',
           'G_J_per_mol_dry_C':np.nan,'Balance_residual':np.nan,'Charge_residual':np.nan,
           'Gas_C_percent':np.nan,'Iterations':np.nan,'EOS_messages':''}
    point_start = time.perf_counter()
    try:
        result = model.solve_equilibrium(case)
        if use_previous and result['phase_allocation'].get('initial_composition_reset'):
            raise RuntimeError('The solver discarded a continuation guess; check the allocator installation.')
        if result['balance_residual'] > BALANCE_TOLERANCE or abs(result['charge_residual']) > BALANCE_TOLERANCE:
            raise RuntimeError('Final elemental/charge/carbon residual exceeds acceptance tolerance.')
        if not np.isfinite(result['gibbs_J_per_mol_feed_C']):
            raise RuntimeError('Nonfinite final Gibbs energy.')
        achieved = result['gas_carbon_fraction_actual']
        if CARBON_MODE=='minimum' and achieved < GAS_CARBON_FRACTION-BALANCE_TOLERANCE:
            raise RuntimeError('Gas-carbon lower bound violated.')
        if CARBON_MODE=='exact' and not np.isclose(achieved,GAS_CARBON_FRACTION,atol=BALANCE_TOLERANCE,rtol=0):
            raise RuntimeError('Gas-carbon equality violated.')
        if any(not np.isfinite(v) or v < -BALANCE_TOLERANCE for v in result['amounts'].values()):
            raise RuntimeError('Invalid final species amount.')
        case_results.append(result)
        previous = result  # Update only after a successful, validated solution.
        row.update(Status='solved',G_J_per_mol_dry_C=result['gibbs_J_per_mol_feed_C'],
                   Balance_residual=result['balance_residual'],Charge_residual=result['charge_residual'],
                   Gas_C_percent=100*achieved,Iterations=result['iterations'],
                   EOS_messages='; '.join(result['eos_trial_errors']))
    except Exception as exc:
        row['Error'] = str(exc)
        # Strict continuation: do not retry from an independent start here.
        # The next point retains the last successful LOWER-temperature seed.
    row['Elapsed_s'] = time.perf_counter()-point_start
    diagnostic_rows.append(row)
    print(f"{temperature:g} C, {case['pressure_bar']:g} bar: {row['Status']}; {start_kind}",flush=True)

diagnostics = pd.DataFrame(diagnostic_rows)
with pd.option_context('display.max_rows',None,'display.max_colwidth',None):
    display(diagnostics)
print(f'Solved {len(case_results)}/{len(temperature_grid)} points in {time.perf_counter()-start_time:.1f} s.')
# Below: failed points remain NaN in all plotted quantities.

## Result tables and plotting helpers

`case_results` contains complete successful solver results; `diagnostics` contains every requested point including failures. `species_results` reports mol/kg and g/kg **original dry feed**, and aqueous-ion molality (mol/kg aqueous solvent water). `carbon_allocation` reports each phase's share of total feed carbon, including salt carbon; its sum should be 100%. Failed temperatures remain NaN. Aqueous absence gives zero aqueous yield at successful points, and undefined ion molality.

Formula legends come from the registries; ion charges are added explicitly. Molecular water is separated in the species plots so it does not hide trace solutes. Hydrate-bound water is part of the solid formula/mass, not part of the molecular-water panel.

In [ ]:
# Quantities are normalized to ORIGINAL dry feed; salt additions are extra mass.
rows, carbon_rows = [], []
for result in case_results:
    factor = result['feed_C_mol_per_g_dry']*1000.  # mol/mol dry C -> mol/kg dry feed
    water_mol = result['amounts'].get(('aqueous','water'),0.)
    water_kg = water_mol*(2*model.AW['H']+model.AW['O'])/1000.
    feed_C = result['feed_atoms'][result['elements'].index('C')]
    per_phase = {p:0. for p in ['gas','aqueous','organic liquid','solid']}
    for (phase,species),amount in result['amounts'].items():
        entry = registry[species]
        atoms_C = entry['elements'].get('C',0)
        mw = sum(model.AW[e]*count for e,count in entry['elements'].items())
        is_ion = phase=='aqueous' and species in model.ION_DATA
        group = ('aqueous ions' if is_ion else 'aqueous neutrals' if phase=='aqueous' else phase)
        rows.append({'T_C':result['temperature_C'],'P_bar':result['pressure_bar'],
                     'Phase':phase,'Group':group,'Species':species,
                     'Formula':entry.get('formula',species),
                     'Amount_mol_per_dry_C':amount,'Yield_mol_per_kg':amount*factor,
                     'Yield_g_per_kg':amount*factor*mw,
                     'Carbon_mol_per_kg':amount*factor*atoms_C,
                     'Ion_molality':amount/water_kg if is_ion and water_kg>0 else np.nan})
        per_phase[phase] += amount*atoms_C
    carbon_rows.append({'T_C':result['temperature_C'],
        **{p:100*value/feed_C for p,value in per_phase.items()},
        'Total_percent':100*sum(per_phase.values())/feed_C,
        'Residual_mol_per_dry_C':sum(per_phase.values())-feed_C})

species_results = pd.DataFrame(rows,columns=['T_C','P_bar','Phase','Group','Species','Formula',
    'Amount_mol_per_dry_C','Yield_mol_per_kg','Yield_g_per_kg','Carbon_mol_per_kg','Ion_molality'])
carbon_allocation = (pd.DataFrame(carbon_rows).set_index('T_C').reindex(run_grid)
                     if carbon_rows else pd.DataFrame(index=run_grid))
carbon_allocation.index.name = 'T_C'
print('Carbon allocation [% of total feed carbon, including salt carbon]:')
display(carbon_allocation)
print('Aqueous ions above the plotting threshold:')
ion_results = species_results[(species_results.Group=='aqueous ions') &
                              (species_results.Yield_mol_per_kg>PLOT_MIN_YIELD)]
display(ion_results[['T_C','Species','Formula','Yield_mol_per_kg','Ion_molality']])


def formula_label(species):
    entry = registry[species]
    formula = entry.get('formula',species)
    if species in model.ION_DATA:
        charge = int(entry['charge'])
        if len(entry['elements'])==1 and next(iter(entry['elements'].values()))==1:
            formula = next(iter(entry['elements']))
        else:
            # Preserve atom counts: HCO3- -> HCO3, CO3^2- -> CO3.
            formula = re.sub(r'\^\d+[+-]$','',formula)
            formula = re.sub(r'[+-]+$','',formula)
        return f"{formula} ({abs(charge) if abs(charge)>1 else ''}{'+' if charge>0 else '−'})"
    return formula.replace('*','·')


def profile_table(group, quantity='Yield_mol_per_kg'):
    part = species_results[species_results.Group.eq(group)]
    if part.empty:
        return pd.DataFrame(index=run_grid)
    table = part.pivot_table(index='T_C',columns='Species',values=quantity,aggfunc='sum').reindex(run_grid)
    for temperature in solved_temperatures:
        table.loc[temperature] = table.loc[temperature].fillna(0.)
    return table


def decorate_axis(ax,title,ylabel):
    ax.set(title=title,xlabel='Temperature [°C]',ylabel=ylabel)
    ax.set_ylim(bottom=0)
    low, high = min(run_grid), max(run_grid)
    ax.set_xlim((low,high) if high>low else (low-5,high+5))
    ax.grid(alpha=.25)
    cutoff = run_config.get('aqueous_cutoff_C')
    if cutoff is not None and run_config.get('aqueous_enabled',True):
        ax.axvline(cutoff,color='black',ls='--',lw=1,alpha=.5)


def save_figure(fig,stem):
    if SAVE_FIGURES:
        FIGURE_DIRECTORY.mkdir(parents=True,exist_ok=True)
        fig.savefig(FIGURE_DIRECTORY/f'{stem}.png',dpi=300,bbox_inches='tight')


solved_temperatures = {r['temperature_C'] for r in case_results}

## Species profiles in every phase

The panels show gas, aqueous neutral species, aqueous ions, organic liquid, solids and molecular water. Yield is mol/kg original dry feed. Use PLOT_TOP_SPECIES=None to show every species above PLOT_MIN_YIELD; use a linear scale if preferred. Symmetric-log axes display zeros while spreading trace yields. Dashed lines mark the stored run's cut-off, not a calculated phase boundary.

In [ ]:
if species_results.empty:
    print('No successful results to plot; inspect diagnostics.')
else:
    fig,axes = plt.subplots(3,2,figsize=(15,13),layout='constrained')
    panels = [('gas','Gas / supercritical fluid'),('aqueous neutrals','Aqueous neutral species'),
              ('aqueous ions','Aqueous ions'),('organic liquid','Organic liquid'),('solid','Solids')]
    for ax,(group,title) in zip(axes.flat,panels):
        table = profile_table(group)
        ranking = table.max().drop(labels=['water'],errors='ignore').sort_values(ascending=False)
        selected = ranking[ranking>PLOT_MIN_YIELD].index.tolist()
        if PLOT_TOP_SPECIES is not None:
            selected = selected[:int(PLOT_TOP_SPECIES)]
        labels = [formula_label(s) for s in selected]
        for species,label in zip(selected,labels):
            ax.plot(table.index,table[species],'-o',ms=3,
                    label=f'{label} [{species}]' if labels.count(label)>1 else label)
        if selected:
            if SPECIES_Y_SCALE=='symlog':
                ax.set_yscale('symlog',linthresh=PLOT_MIN_YIELD)
            ax.legend(fontsize=8)
        else:
            ax.text(.5,.5,'No non-water species above threshold',transform=ax.transAxes,ha='center',va='center')
        decorate_axis(ax,title,'Species yield [mol/kg dry feed]')
    ax=axes.flat[5]
    for group,label in [('gas','H2O — gas'),('aqueous neutrals','H2O — aqueous'),('organic liquid','H2O — organic liquid')]:
        table = profile_table(group)
        if 'water' in table:
            ax.plot(table.index,table.water,'-o',ms=3,label=label)
    decorate_axis(ax,'Molecular water distribution','Water [mol/kg dry feed]')
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)
    fig.suptitle(f"{run_name}: species by phase\nFormula legends; water separate; dashed line = imposed aqueous cut-off; gaps = failed points")
    save_figure(fig,'species_by_phase')
    plt.show()

## Carbon allocation, char and mineral yields

Carbon fractions count carbon atoms, so one mole of benzene contributes six moles of carbon. Carbon in carbonate minerals counts toward solid carbon. Under an exact gas-carbon constraint, the remaining carbon may be forced into other permitted phases; a cut-off can therefore create a jump in char. Mineral masses include crystal water. Ash entered as unmodelled bookkeeping is not included in predicted solid yields.

In [ ]:
if species_results.empty:
    print('No successful results to plot.')
else:
    fig,axes=plt.subplots(1,2,figsize=(14,5),layout='constrained')
    for phase in ['gas','aqueous','organic liquid','solid']:
        axes[0].plot(carbon_allocation.index,carbon_allocation[phase],'-o',ms=3,label=phase)
    mode = run_controls['carbon_mode']
    if mode in ('minimum','exact'):
        axes[0].axhline(100*run_controls['gas_fraction'],color='grey',ls=':',lw=1,label=f'{mode} gas-carbon target')
    decorate_axis(axes[0],'Carbon allocation between phases','Carbon [% of total feed carbon]')
    axes[0].legend(fontsize=8)
    table=profile_table('solid','Yield_g_per_kg')
    if 'carbon' in table:
        axes[1].plot(table.index,table['carbon'],'-o',ms=3,label='C — char')
        axes[1].legend()
    decorate_axis(axes[1],'Char yield','Char [g/kg original dry feed]')
    fig.suptitle(f'{run_name}: carbon accounting (includes added-salt carbon)')
    save_figure(fig,'carbon_allocation_and_char')
    plt.show()

    # Separate char from minerals so trace mineral/hydrate yields are readable.
    fig,ax=plt.subplots(figsize=(10,5),layout='constrained')
    plotted=False
    for species in table.columns:
        if species!='carbon' and table[species].max()>1e-8:
            ax.plot(table.index,table[species],'-o',ms=3,label=formula_label(species))
            plotted=True
    decorate_axis(ax,'Mineral and hydrate yields','Solid [g/kg original dry feed]')
    if plotted:
        ax.legend(fontsize=8)
    else:
        ax.text(.5,.5,'No mineral yield above threshold',transform=ax.transAxes,ha='center',va='center')
    save_figure(fig,'mineral_yields')
    plt.show()

## Optional comparison with Figures 7 and 8

This cell contains the same **approximate digitizations of calculated equilibrium curves** used in the earlier Yakaboylu et al. (2013) comparison. They are not experimental observations and have not been re-digitized here. Reference conditions: 240 bar and the original reduced manure feed/water ratio. Figure 7 compares major/minor gas yields; Figure 8 compares gaseous/liquid molecular water, excluding hydrate-bound water.

The default current case adds salts, offers additional molecules and uses an imposed 400 °C cut-off. Interpret agreement as a trend comparison rather than an identical reproduction of the published calculation. If UCA, pressure, water ratio or constraints change, the reference markers remain fixed.

In [ ]:
# Optional trend comparison; no earlier HTG_Cases notebook cells are required.
# Approximate digitizations retained from the original Figures 7–8 comparison.
# These reference values are published-model curves, NOT experimental data.
TC = np.array([100,150,200,250,300,340,360,380,389.6,390,400,420,460,500,540,580],dtype=float)
paper78 = pd.DataFrame({
 'T_C':TC,
 'CO2':[1.3,2.2,2.8,3.2,4.3,14.4,14.7,15.0,15.0,15.0,15.0,15.1,15.4,15.8,16.2,16.7],
 'CH4':[0,0,0,0,.2,9.2,9.5,9.9,9.9,9.9,9.9,9.8,9.5,9.1,8.6,8.0],
 'H2':[0,0,0,0,0,0,.3,1.1,1.2,1.2,1.3,1.8,2.9,4.3,6.4,8.9],
 'N2':[.75,.78,.75,.70,.70,.89,.91,.93,.93,.93,.93,.93,.93,.92,.92,.91],
 'NH3':[0,0,0,0,0,0,.03,.04,.04,.04,.04,.05,.05,.06,.06,.07],
 'H2S':[0,0,0,0,.05,.03,.02,.06,.16,.16,.16,.16,.16,.16,.16,.16],
 'CO':[0,0,0,0,0,0,0,0,0,0,.01,.01,.04,.08,.16,.28],
 'H2O_g':[0,0,0,1,5,55,142,218,223,223,223,222,222,221,220,219],
 'H2O_l':[242,242,241,240,238,167,106,5,0,0,0,0,0,0,0,0],
})
if not case_results:
    print('No successful current-case results to compare.')
else:
    mapping={'CO2':'carbon dioxide','CH4':'methane','H2':'hydrogen','N2':'nitrogen',
             'NH3':'ammonia','H2S':'hydrogen sulfide','CO':'carbon monoxide'}
    reference_rows=[]
    for result in case_results:
        factor=result['feed_C_mol_per_g_dry']*1000.
        amounts=result['amounts']
        reference_rows.append({'T_C':result['temperature_C'],
            **{label:amounts.get(('gas',species),0.)*factor for label,species in mapping.items()},
            'H2O_g':amounts.get(('gas','water'),0.)*factor,
            'H2O_l':sum(amounts.get((phase,'water'),0.) for phase in ['aqueous','organic liquid'])*factor})
    figure78_results=pd.DataFrame(reference_rows).set_index('T_C').reindex(run_grid)
    fig,axes=plt.subplots(1,2,figsize=(15,5),layout='constrained')
    for ax,columns,title in [(axes[0],['CO2','CH4','H2'],'Figure 7: major gases'),
                              (axes[1],['N2','NH3','H2S','CO'],'Figure 7: minor gases')]:
        for column in columns:
            line,=ax.plot(figure78_results.index,figure78_results[column],'-',lw=1.8,label=f'{column}: current model')
            ax.plot(paper78.T_C,paper78[column],'o',ms=4,mfc='none',color=line.get_color(),label=f'{column}: reference')
        decorate_axis(ax,title,'Gas yield [mol/kg dry feed]')
        ax.legend(fontsize=8)
    fig.suptitle(f'{run_name}: Figure 7 comparison\nReference: approximate digitized Yakaboylu et al. (2013) model curves')
    save_figure(fig,'figure7_comparison')
    plt.show()
    fig,ax=plt.subplots(figsize=(10,5),layout='constrained')
    for column,label in [('H2O_g','H2O — gas'),('H2O_l','H2O — liquid')]:
        line,=ax.plot(figure78_results.index,figure78_results[column],'-',lw=1.8,label=f'{label}: current model')
        ax.plot(paper78.T_C,paper78[column],'o',ms=4,mfc='none',color=line.get_color(),label=f'{label}: reference')
    decorate_axis(ax,'Figure 8: water distribution','Water [mol/kg dry feed]')
    ax.legend(fontsize=9)
    fig.suptitle(f'{run_name}: Figure 8 comparison')
    save_figure(fig,'figure8_comparison')
    plt.show()
    print('Reference conditions: 240 bar, original reduced dry-manure feed and water ratio.')
    print('Current pressures [bar]:', sorted(set(diagnostics.P_bar)))
    print('Current total added water [g/g dry feed]:',run_config['water_g_per_g_dry_feed'])
    print('Current salt additions [mol/g dry feed]:',
          {name:salt['mol_per_g_dry_feed'] for name,salt in run_config['added_salts'].items()})
    print('Use this as a trend comparison when feed, species, constraints or pressure differ. Liquid water excludes hydrate-bound water.')